In [ ]:
# IBR_STARTUP

<!-- IBR_ETAPA:1|Entendo sobre IBR -->

Conteudo para expolicar a origem dos dados etc

<!-- IBR_ETAPA:2|Preparando e entedendo os dados coletados -->

## 1. Escolha dos dados

- `MODO = "amostra"` 
- `MODO = "dataset"` 


In [1]:
from pathlib import Path
import shutil, subprocess, time, re, os

PROJECT_DIR = Path.cwd()
if PROJECT_DIR.name == "notebooks":
    PROJECT_DIR = PROJECT_DIR.parent

MODO = "dataset" 
SAMPLE_DIR = PROJECT_DIR / "data" / "sample"
DATASET_DIR = PROJECT_DIR / "data" / "raw"
SCRIPT = PROJECT_DIR / "scripts" / "aggregate_pcaps.sh"
DATA_DIR = SAMPLE_DIR if MODO == "amostra" else DATASET_DIR

print("Modo:", MODO)
print("Dados:", DATA_DIR)

Modo: dataset
Dados: d:\projeto jupyter notebook\data\raw


## 2. Conferir arquivos

Contamos total de arquivos, PCAPs, TXT e tamanho total.

In [ ]:
from pathlib import Path
import zipfile

ZIP_DATASET = Path(r"/home/jovyan/ibr-notebook/data/raw/cloud_telescope_raw_dataset_3.zip")

if not ZIP_DATASET.exists():
    raise FileNotFoundError(
        f"Arquivo não encontrado: {ZIP_DATASET}"
    )

if not zipfile.is_zipfile(ZIP_DATASET):
    raise ValueError(
        f"O arquivo existe, mas não é um ZIP válido: {ZIP_DATASET}"
    )

print("Dataset encontrado:")
print(ZIP_DATASET)

print(f"Tamanho do ZIP: {ZIP_DATASET.stat().st_size / (1024**3):.2f} GiB")

with zipfile.ZipFile(ZIP_DATASET, "r", allowZip64=True) as z:
    arquivos = [
        info for info in z.infolist()
        if not info.is_dir()
    ]

    pcaps = [
        info for info in arquivos
        if info.filename.lower().endswith(".pcap.gz")
    ]

    txts = [
        info for info in arquivos
        if info.filename.lower().endswith(".txt")
    ]

print("Total de arquivos:", len(arquivos))
print("PCAPs:", len(pcaps))
print("TXT:", len(txts))

with zipfile.ZipFile(ZIP_DATASET, "r", allowZip64=True) as z:
    arquivos = [
        info for info in z.infolist()
        if not info.is_dir()
    ]

## 3. Visualizar nomes

Os nomes ajudam a identificar zona/região, IP público e data/hora.

In [ ]:
for p in arquivos[::1000]:
    print(p.filename)

## 4. Abrir um metadado TXT

In [ ]:
import pyzipper

SENHA = b"Radiation!"

with pyzipper.AESZipFile(ZIP_DATASET, "r") as z:
    z.setpassword(SENHA)

    txt = txts[1000]

    print("Arquivo:", txt.filename)
    print("-" * 60)

    conteudo = z.read(txt.filename).decode(
        "utf-8",
        errors="replace"
    )

    print(conteudo[:3000])

## 5. Ver alguns pacotes

Usa `tshark` somente nos primeiros 5 pacotes.

In [ ]:
from pathlib import Path
import pyzipper
import tempfile
import subprocess
import shutil
import pandas as pd
from io import StringIO

TSHARK = shutil.which("tshark")

SENHA = b"Radiation!"

indice = 5
pcap = pcaps[indice]

print("PCAP escolhido:", pcap.filename)

with tempfile.TemporaryDirectory() as pasta_temp:

    pasta_temp = Path(pasta_temp)

    arquivo_temp = pasta_temp / Path(pcap.filename).name

    with pyzipper.AESZipFile(ZIP_DATASET, "r") as z:
        z.setpassword(SENHA)

        with z.open(pcap.filename) as origem:
            with open(arquivo_temp, "wb") as destino:
                shutil.copyfileobj(origem, destino)

    cmd = [
        TSHARK,
        "-r", str(arquivo_temp),
        "-c", "10",
        "-T", "fields",
        "-E", "header=y",
        "-E", "separator=,",
        "-e", "ip.src",
        "-e", "ip.dst",
        "-e", "_ws.col.Protocol",
        "-e", "tcp.dstport",
        "-e", "udp.dstport"
    ]

    resultado = subprocess.run(
        cmd,
        capture_output=True,
        text=True,
        errors="replace"
    )

    if resultado.returncode == 0 and resultado.stdout.strip():

        df = pd.read_csv(StringIO(resultado.stdout))

        df = df.rename(columns={
            "ip.src": "IP de origem",
            "ip.dst": "IP de destino",
            "_ws.col.Protocol": "Protocolo",
            "tcp.dstport": "Porta TCP",
            "udp.dstport": "Porta UDP"
        })

        display(df)

    else:
        print("Erro ao processar o PCAP:")
        print(resultado.stderr)

In [ ]:
from pathlib import Path
import os
import re
import shutil
import subprocess
import zipfile

import pandas as pd
import pyzipper

INICIO = pd.Timestamp("2023-12-10 00:00")
FIM = pd.Timestamp("2023-12-16 23:59")

ZIP_DATASET = Path(os.environ["DATASET_ZIP_PATH"])
SENHA = os.environ["DATASET_ZIP_PASSWORD"].encode()

WORK_DIR = Path("/tmp/ibr_periodo")
RAW_DIR = WORK_DIR / "raw"

OUTPUT_DIR = (
    Path("/home/jovyan/ibr-notebook/data/periodos")
    / f"{INICIO:%Y%m%d_%H%M}__{FIM:%Y%m%d_%H%M}"
)

print("Período:", INICIO, "até", FIM)

In [ ]:
timestamp_re = re.compile(r"-(\d{8})-(\d{4})\.pcap\.gz$")

selecionados = []

with zipfile.ZipFile(ZIP_DATASET, "r", allowZip64=True) as z:
    for info in z.infolist():
        if not info.filename.lower().endswith(".pcap.gz"):
            continue

        nome = Path(info.filename).name
        match = timestamp_re.search(nome)

        if not match:
            continue

        timestamp = pd.to_datetime(
            match.group(1) + match.group(2),
            format="%Y%m%d%H%M"
        )

        if INICIO <= timestamp <= FIM:
            selecionados.append(info)

print("PCAPs selecionados:", len(selecionados))
print(
    "Tamanho comprimido:",
    f"{sum(p.file_size for p in selecionados) / (1024**3):.3f} GiB"
)

In [ ]:
if WORK_DIR.exists():
    shutil.rmtree(WORK_DIR)

RAW_DIR.mkdir(parents=True, exist_ok=True)

with pyzipper.AESZipFile(ZIP_DATASET, "r") as z:
    z.setpassword(SENHA)

    for i, info in enumerate(selecionados, start=1):
        destino = RAW_DIR / Path(info.filename).name

        with z.open(info.filename) as origem, open(destino, "wb") as saida:
            shutil.copyfileobj(origem, saida)

        if i % 100 == 0 or i == len(selecionados):
            print(f"Extraídos: {i}/{len(selecionados)}")

<!-- IBR_ETAPA:3|Preparação e agregação dos PCAPs -->

In [ ]:
from tqdm.auto import tqdm
import subprocess
import os
from pathlib import Path

SCRIPT = Path(
    "/home/jovyan/ibr-notebook/scripts/aggregate_pcaps.sh"
)

env = os.environ.copy()

env["KEEP_INTERMEDIATE"] = "0"

processo = subprocess.Popen(
    [
        "bash",
        str(SCRIPT),
        str(RAW_DIR),
        "aggregated",
    ],
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

barra = None
etapa_atual = None

nomes_etapas = {
    "1": "Reescrevendo IPs de destino",
    "2": "Separando por região AWS",
    "3": "Unindo PCAPs por região",
    "4": "Comprimindo resultados",
}

for linha in processo.stdout:

    linha = linha.strip()

    # Início de uma nova etapa
    if linha.startswith("STAGE|"):

        _, etapa, descricao, total = linha.split("|")

        if barra is not None:
            barra.close()

        etapa_atual = etapa

        print(
            f"\nEtapa {etapa}/4 - {descricao}"
        )

        barra = tqdm(
            total=int(total),
            unit="arquivo",
            leave=True
        )

    # Atualização de progresso
    elif linha.startswith("PROGRESS|"):

        _, etapa, atual, total = linha.split("|")

        atual = int(atual)

        if barra is not None:
            barra.n = atual
            barra.refresh()

    elif linha == "DONE":

        if barra is not None:
            barra.close()

processo.wait()

if processo.returncode != 0:
    raise RuntimeError(
        "Erro durante a agregação."
    )

print("\nAgregação concluída.")

In [ ]:
AGG_TEMP = RAW_DIR / "aggregated"
AGG_FINAL = OUTPUT_DIR / "aggregated"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if AGG_FINAL.exists():
    shutil.rmtree(AGG_FINAL)

shutil.copytree(AGG_TEMP, AGG_FINAL)

agregados = sorted(AGG_FINAL.glob("*.pcap.gz"))

print("Arquivos agregados:", len(agregados))
print("Pasta final:", AGG_FINAL)

shutil.rmtree(WORK_DIR)

print("Arquivos temporários removidos.")

In [ ]:
for arquivo in agregados:
    print(
        f"{arquivo.name:<30}"
        f"{arquivo.stat().st_size / (1024**2):8.2f} MiB"
    )

<!-- IBR_ETAPA:4|Extração com TShark -->

In [ ]:
from pathlib import Path
import subprocess

SCRIPT_TSHARK = Path(
    "/home/jovyan/ibr-notebook/scripts/1_generate_tshark_csv.sh"
)

subprocess.run(
    [
        "bash",
        str(SCRIPT_TSHARK)
    ],
    cwd=AGG_FINAL,
    check=True
)

CSV_GZ = AGG_FINAL / "output.csv.gz"

print("CSV gerado:")
print(CSV_GZ)

<!-- IBR_ETAPA:5|Enriquecimento -->

In [ ]:
import time

inicio = time.perf_counter()

SCRIPT_ENRICH = Path(
    "/home/jovyan/ibr-notebook/scripts/2_enrich_csv.py"
)

comando = f'''
set -o pipefail

zcat output.csv.gz \
| python3 "{SCRIPT_ENRICH}" \
| gzip > output.csv.enriched.gz
'''

subprocess.run(
    [
        "bash",
        "-lc",
        comando
    ],
    cwd=AGG_FINAL,
    check=True
)

tempo = time.perf_counter() - inicio

CSV_ENRICHED = (
    AGG_FINAL
    / "output.csv.enriched.gz"
)

print("CSV enriquecido gerado:")
print(CSV_ENRICHED)

print(
    f"Tempo de enriquecimento: "
    f"{tempo / 60:.2f} minutos"
)

In [ ]:
import gzip

with gzip.open(
    CSV_ENRICHED,
    "rt",
    encoding="utf-8",
    errors="replace"
) as f:
    for _ in range(5):
        print(f.readline().strip())

<!-- IBR_ETAPA:6|Sumarização do csv-->

In [ ]:
SCRIPT_SUMMARY = Path(
    "/home/jovyan/ibr-notebook/scripts/3_summarize_radiation.py"
)

ANALYSIS_DIR = (
    AGG_FINAL
    / "analysis"
)

comando = f'''
set -o pipefail

zcat output.csv.enriched.gz \
| python3 "{SCRIPT_SUMMARY}" "{ANALYSIS_DIR}"
'''

subprocess.run(
    [
        "bash",
        "-lc",
        comando
    ],
    cwd=AGG_FINAL,
    check=True
)

print("Análises geradas em:")
print(ANALYSIS_DIR)

<!-- IBR_ETAPA:7|Tabelas e graficos -->

In [ ]:
import pandas as pd

tcp_ports = pd.read_csv(
    ANALYSIS_DIR / "tcp_ports.csv"
)

tabela_portas = (
    tcp_ports
    .head(10)
    [["tcp_dstport", "percentage"]]
    .copy()
)

tabela_portas.columns = [
    "Porta TCP",
    "Pacotes (%)"
]

tabela_portas["Porta TCP"] = (
    tabela_portas["Porta TCP"]
    .astype(int)
)

tabela_portas["Pacotes (%)"] = (
    tabela_portas["Pacotes (%)"]
    .round(1)
)

tabela_portas.index = range(
    1,
    len(tabela_portas) + 1
)

tabela_portas["Pacotes (%)"] = (
    tabela_portas["Pacotes (%)"]
    .map(lambda x: f"{x:.1f}%".replace(".", ","))
)
tabela_portas

In [1]:
# IBR_ONSTART
from ibr_aprendizado import botao_reset

botao_reset()

In [2]:
# IBR_ONSTART
from ibr_aprendizado import pergunta

pergunta("memcached_port")